# 3D viewer for one NIfTI file

> **Archived**: true 3D viewing is not in use for now; see `archive/README.md`. Use `notebooks/view_slices.ipynb` / `compare_slices.ipynb` instead.

Pick any `.nii` / `.nii.gz` file under `data/` or `runs/` and view it in 3D.

- **Image**: an MRI scan, shown in one of three ways (**Show as**):
  - **3D volume** (default): the whole scan rendered see-through; brighter tissue is more opaque.
  - **3D surface**: a solid surface through the scan at one intensity, e.g. the outside of the head or the brain.
  - **Slices**: three grey planes (sagittal, coronal, axial) that you move with the sliders.

  **Level %** sets the intensity, as a percentile of the tissue voxels. For the volume it hides everything darker; for the surface it is the intensity the surface is drawn at. Raise it to see inside. If you pick a label file as the image, it is shown as coloured surfaces instead.
- **Mask** (optional): a label or prediction file, drawn as coloured surfaces. Picking an image from an `imagesTr/` folder fills in the matching label automatically.
  - anterior/posterior (MSD, predictions): orange = anterior, blue = posterior
  - subfields (external dataset): gold = CA1-3, green = subiculum, purple = CA4-DG (names inferred from the atlas)
- **Moving the view**: drag to rotate, scroll to zoom, right-drag to pan. Click a legend entry to hide or show it.

The Level and slice sliders update the figure in place, keeping your view, when `anywidget` is installed (`pip install anywidget`, then restart the kernel). Without it the figure is redrawn when you release a slider and the view resets.

Large scans are shown at reduced resolution: up to 64 voxels per side for the volume and 128 for surfaces and slices.

Run all cells, then use the controls under the last cell. Research use only, not for diagnosis.

In [ ]:
# Work from the project root (so data/ and runs/ resolve) and import hippo from src/
import math
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hippo").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "archive"))  # render3d.py is archived
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import ipywidgets as w
import numpy as np
import plotly.graph_objects as go
import vedo
from IPython.display import clear_output, display

import render3d

try:
    import anywidget  # noqa: F401  lets go.FigureWidget move slices in place without resetting the view
    LIVE = True
except ImportError:
    LIVE = False
print(f"project: {ROOT}\nslice sliders: {'live (view kept)' if LIVE else 'redraw on release (pip install anywidget for live)'}")

In [ ]:
SEARCH_DIRS = ["data", "runs"]
MAX_DISPLAY = 128       # longest side of the MRI grid for slices and surfaces; larger scans are subsampled
MAX_VOLUME = 64         # longest side of the grid for volume rendering (the browser slows down above ~300k points)
MAX_TRIANGLES = 60_000  # per surface; finer meshes are simplified so the browser stays responsive
NONE = "(none)"
STYLES = [("3D volume", "volume"), ("3D surface", "surface"), ("Slices", "slices")]

# label value -> (name, colour), chosen by which values a mask contains
LABEL_SETS = [
    ({1, 2}, {1: ("anterior", "orangered"), 2: ("posterior", "steelblue")}),
    ({1, 2, 3}, {1: ("CA1-3", "gold"), 2: ("subiculum", "mediumseagreen"), 3: ("CA4-DG", "orchid")}),
]
PALETTE = ["orangered", "steelblue", "gold", "mediumseagreen", "orchid", "turquoise", "salmon", "khaki"]


def list_nifti():
    return sorted(str(p) for d in SEARCH_DIRS for p in Path(d).rglob("*") if p.name.endswith((".nii.gz", ".nii")))


def guess_mask(image):
    """The label file next to an image in an imagesTr/ or imagesTs/ folder (MSD, external crops), if any."""
    p = Path(image)
    for images, labels in [("imagesTr", "labelsTr"), ("imagesTs", "labelsTs")]:
        if p.parent.name == images and (p.parent.parent / labels / p.name).exists():
            return str(p.parent.parent / labels / p.name)
    return NONE


def load(path):
    """Array in RAS orientation, voxel size (mm) and world position of the first voxel."""
    data, spacing, origin = render3d.load_ras(str(path))
    return np.asarray(data, dtype=np.float32), np.array(spacing), np.array(origin)


def subsample(data, spacing, max_side):
    """Every n-th voxel so the longest side is at most max_side; returns data and the new voxel size."""
    step = max(1, math.ceil(max(data.shape) / max_side))
    return data[::step, ::step, ::step], spacing * step


def is_label_map(data):
    """Whole numbers with only a few distinct values: a mask, not an MRI."""
    sample = data[::4, ::4, ::4]
    return bool(np.all(sample == np.rint(sample))) and len(np.unique(sample)) <= 20


def label_colours(values):
    for known, names in LABEL_SETS:
        if values and values <= known:
            return {v: names[v] for v in sorted(values)}
    return {v: (f"label {v}", PALETTE[i % len(PALETTE)]) for i, v in enumerate(sorted(values))}


def mesh_trace(mesh, name, colour, opacity=1.0):
    if mesh.ncells > MAX_TRIANGLES:
        mesh = mesh.decimate(fraction=MAX_TRIANGLES / mesh.ncells)
    v, f = mesh.vertices, np.asarray(mesh.cells)
    return go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2], i=f[:, 0], j=f[:, 1], k=f[:, 2], color=colour,
                     opacity=opacity, name=name, showlegend=True, hoverinfo="name",
                     lighting=dict(ambient=0.45, diffuse=0.8, specular=0.25, roughness=0.6))


def mask_traces(path):
    """One coloured surface per label value, and the world centre (mm) of all labelled voxels."""
    data, spacing, origin = load(path)
    data = np.rint(data).astype(np.int32)
    if not (data > 0).any():
        return [], None
    traces = []
    for value, (name, colour) in label_colours(set(np.unique(data).tolist()) - {0}).items():
        if (data == value).sum() >= 2:
            mesh = vedo.Volume((data == value).astype(np.uint8), spacing=spacing, origin=origin) \
                .isosurface(0.5).smooth().triangulate()
            traces.append(mesh_trace(mesh, f"{Path(path).name}: {name}", colour))
    return traces, origin + np.argwhere(data > 0).mean(0) * spacing


class Mri3D:
    """The MRI in 3D: a translucent volume rendering, or a solid surface at one intensity level.

    `level` is a percentile (0-100) of the tissue intensities (voxels above the background value):
    the volume shows everything brighter than it, the surface is drawn at exactly that intensity.
    """

    def __init__(self, data, spacing, origin, style, see_through=False):
        self.style, self.origin, self.see_through = style, origin, see_through
        self.data, self.spacing = subsample(data, spacing, MAX_VOLUME if style == "volume" else MAX_DISPLAY)
        tissue = self.data[self.data > self.data.min()]
        self.levels = np.percentile(tissue if tissue.size else self.data, np.arange(101))

    def trace(self, level):
        if self.style == "volume":
            grid = np.meshgrid(*[self.origin[a] + np.arange(self.data.shape[a]) * self.spacing[a] for a in range(3)],
                               indexing="ij")
            return go.Volume(x=grid[0].ravel(), y=grid[1].ravel(), z=grid[2].ravel(), value=self.data.ravel(),
                             isomin=self.levels[level], isomax=self.levels[100], opacity=0.12, surface_count=15,
                             colorscale="gray", showscale=False, name="MRI volume", showlegend=True, hoverinfo="skip")
        mesh = vedo.Volume(self.data, spacing=self.spacing, origin=self.origin).isosurface(self.levels[level])
        if mesh.npoints == 0:
            raise ValueError("no surface at this level; try another Level")
        # see-through when a mask is inside, so the hippocampus stays visible
        return mesh_trace(mesh.triangulate(), "MRI surface", "lightgray", 0.35 if self.see_through else 1.0)

    def update(self, level):
        """Changes for the existing trace when the level slider moves."""
        if self.style == "volume":
            return dict(isomin=self.levels[level])
        new = self.trace(level)
        return dict(x=new.x, y=new.y, z=new.z, i=new.i, j=new.j, k=new.k)


class SliceView:
    """Three orthogonal grey MRI planes that can each be moved along their axis."""
    AXES = ["Sagittal x", "Coronal y", "Axial z"]

    def __init__(self, data, spacing, origin):
        self.data, self.spacing = subsample(data, spacing, MAX_DISPLAY)
        self.coords = [origin[a] + np.arange(self.data.shape[a]) * self.spacing[a] for a in range(3)]
        self.vmin, self.vmax = np.percentile(self.data, [1, 99.5])

    def size(self, axis):
        return len(self.coords[axis])

    def index_of(self, axis, mm):
        return int(np.clip(round((mm - self.coords[axis][0]) / self.spacing[axis]), 0, self.size(axis) - 1))

    def plane(self, axis, i):
        """Coordinates (mm) and grey values of the plane through display voxel i along `axis`."""
        x, y, z = self.coords
        if axis == 0:
            Y, Z = np.meshgrid(y, z, indexing="ij")
            return dict(x=np.full_like(Y, x[i]), y=Y, z=Z, surfacecolor=self.data[i])
        if axis == 1:
            X, Z = np.meshgrid(x, z, indexing="ij")
            return dict(x=X, y=np.full_like(X, y[i]), z=Z, surfacecolor=self.data[:, i])
        X, Y = np.meshgrid(x, y, indexing="ij")
        return dict(x=X, y=Y, z=np.full_like(X, z[i]), surfacecolor=self.data[:, :, i])

    def traces(self, idx):
        return [go.Surface(**self.plane(a, i), colorscale="gray", cmin=self.vmin, cmax=self.vmax, showscale=False,
                           hoverinfo="skip", name=f"MRI {self.AXES[a].split()[0].lower()}", showlegend=True,
                           lighting=dict(ambient=1.0, diffuse=0.0, specular=0.0))
                for a, i in enumerate(idx)]


def build(image, mask=NONE, style="volume", level=40):
    """Figure for an image (+ optional mask).

    Returns (figure, SliceView or None, slice indices, Mri3D or None). The MRI traces come first
    in the figure: three slice planes, or one volume / surface trace.
    """
    data, spacing, origin = load(image)
    traces, centre = mask_traces(mask) if mask != NONE else ([], None)
    view, idx, mri = None, [], None
    if is_label_map(data):  # a label file picked as the image: show it as surfaces
        own, own_centre = mask_traces(image)
        traces = own + traces
    elif style == "slices":
        view = SliceView(data, spacing, origin)
        if centre is None:
            centre = [view.coords[a].mean() for a in range(3)]
        idx = [view.index_of(a, centre[a]) for a in range(3)]
        traces = view.traces(idx) + traces
    else:
        mri = Mri3D(data, spacing, origin, style, see_through=bool(traces))
        traces = [mri.trace(level)] + traces
    if not traces:
        raise ValueError(f"nothing to show in {image}")
    title = Path(image).name + (f"  +  {Path(mask).name}" if mask != NONE else "")
    fig = go.Figure(traces)
    fig.update_layout(
        title=f"{title}<br><sup>drag: rotate · scroll: zoom · right-drag: pan · click legend: show/hide</sup>",
        template="plotly_dark", height=700, margin=dict(l=0, r=0, t=60, b=0),
        scene=dict(aspectmode="data", xaxis_title="R (mm)", yaxis_title="A (mm)", zaxis_title="S (mm)",
                   camera=dict(eye=dict(x=-1.4, y=1.2, z=0.9), up=dict(x=0, y=0, z=1)), uirevision="keep"),
    )
    return fig, view, idx, mri

In [ ]:
files = list_nifti()
wide = w.Layout(width="95%")
search = w.Text(placeholder="type to filter, e.g.  hippocampus_135   or   s03 t1w_standard", description="Filter", layout=wide)
image_dd = w.Dropdown(options=files, description="Image", layout=wide)
mask_dd = w.Dropdown(options=[NONE] + files, description="Mask", layout=wide)
style_tb = w.ToggleButtons(options=STYLES, value="volume", description="Show as")
show_btn = w.Button(description="Show in 3D", button_style="primary", icon="cube")
level_sl = w.IntSlider(value=40, min=0, max=99, description="Level %", continuous_update=False, layout=wide)
sliders = [w.IntSlider(description=name, continuous_update=LIVE, disabled=True, layout=wide)
           for name in SliceView.AXES]
out = w.Output()
state = {"fig": None, "view": None, "mri": None, "ready": False}


def redraw():
    with out:
        clear_output(wait=True)
        display(state["fig"])


def on_filter(change):
    terms = change["new"].lower().split()
    match = [f for f in files if all(t in f.lower() for t in terms)]
    image_dd.options = match
    mask_dd.options = [NONE] + match
    on_image({"new": image_dd.value})


def on_image(change):
    if not change["new"]:
        return
    guess = guess_mask(change["new"])
    if guess not in mask_dd.options:
        mask_dd.options = list(mask_dd.options) + [guess]
    mask_dd.value = guess


def on_show(_=None):
    if not image_dd.value:
        with out:
            clear_output()
            print("No file matches the filter.")
        return
    with out:
        clear_output()
        print(f"Loading {image_dd.value} ...")
    state["ready"] = False
    try:
        fig, view, idx, mri = build(image_dd.value, mask_dd.value, style_tb.value, level_sl.value)
    except Exception as e:  # show the problem under the controls instead of a long traceback
        with out:
            clear_output()
            print(f"Could not show this file: {e}")
        return
    state.update(fig=go.FigureWidget(fig) if LIVE else fig, view=view, mri=mri)
    level_sl.disabled = mri is None
    for axis, slider in enumerate(sliders):
        slider.disabled = view is None
        if view is not None:
            slider.max = view.size(axis) - 1
            slider.value = idx[axis]
    state["ready"] = True
    redraw()


def on_style(change):
    if state["fig"] is not None:  # re-show the current file in the new style
        on_show()


def on_level(change):
    if not state["ready"] or state["mri"] is None:
        return
    try:
        changes = state["mri"].update(change["new"])
    except ValueError as e:
        with out:
            print(e)
        return
    with state["fig"].batch_update():
        state["fig"].data[0].update(**changes)
    if not LIVE:
        redraw()


def on_slide(axis, change):
    fig, view = state["fig"], state["view"]
    if not state["ready"] or view is None:
        return
    with fig.batch_update():
        fig.data[axis].update(**view.plane(axis, change["new"]))
    if not LIVE:
        redraw()


search.observe(on_filter, names="value")
image_dd.observe(on_image, names="value")
style_tb.observe(on_style, names="value")
show_btn.on_click(on_show)
level_sl.observe(on_level, names="value")
for axis, slider in enumerate(sliders):
    slider.observe(lambda change, axis=axis: on_slide(axis, change), names="value")

on_image({"new": image_dd.value})
display(w.VBox([search, image_dd, mask_dd, w.HBox([style_tb, show_btn]), level_sl, *sliders, out]))